In [ ]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import os 
import sys
from pathlib import Path

### Setup Configuration

In [ ]:
ROOT = Path(os.path.abspath('')).parent.parent
print(f"ROOT: {ROOT}")

EVAL_DATA_PATH = ROOT / "data" / "evaluation"
EMBEDDINGS_PATH = ROOT / "data" / "embeddings"
PROCESSED_DATA_PATH = ROOT / "data" / "processed"

EMBEDDINGS_FILE = EMBEDDINGS_PATH / "open_source_embeddings.npy"
EVALUATION_FILE = EVAL_DATA_PATH / "open_ai_evaluation_dataset.json"
CHUNKS_FILE = PROCESSED_DATA_PATH / "all_chunks.jsonl"

MODEL_NAME = "all-MiniLM-L6-v2"


### Load embeddings, chunks, and eval dataset

In [ ]:
all_embeddings = np.load(EMBEDDINGS_FILE)
print(f"All embeddings shape: {all_embeddings.shape}")

In [ ]:
chunks_df = pd.read_json(CHUNKS_FILE, lines=True)
print(f"Chunks dataframe shape: {chunks_df.shape}")
display(chunks_df.head())

In [ ]:
all_texts = chunks_df['page_content'].tolist()
print(f"Total number of text chunks: {len(all_texts)}")

In [ ]:
with open(EVALUATION_FILE, 'r') as f:
    evaluation_data = json.load(f)
print(f"Total number of evaluation question-answer pairs: {len(evaluation_data)}")
display(evaluation_data[:2])

### Select embedding model

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"Using embedding model: {MODEL_NAME}")

### Define evaluation metrics

In [ ]:
def calculate_metrics(retrieved_texts, correct_answer):
    hit = 0
    reciprocal_rank = 0

    if correct_answer in retrieved_texts:
        hit = 1
        rank = retrieved_texts.index(correct_answer) + 1
        reciprocal_rank = 1 / rank
    return hit, reciprocal_rank

### Do Evaluation

In [ ]:
from tqdm import tqdm
total_hits = 0
total_reciprocal_rank = 0

k = 5

for item in tqdm(evaluation_data, desc="Evaluating"):
    query = item['query']
    correct_chunk_index = item['correct_chunk_index']

    query_embedding = model.encode([query])

    similarities = cosine_similarity(query_embedding, all_embeddings)[0]
    top_k_indices = similarities.argsort()[-k:][::-1]

    hit = 0
    reciprocal_rank = 0

    if correct_chunk_index in top_k_indices:
        hit = 1
        rank = list(top_k_indices).index(correct_chunk_index) + 1
        reciprocal_rank = 1 / rank

    total_hits += hit
    total_reciprocal_rank += reciprocal_rank



In [ ]:
num_queries = len(evaluation_data)
hit_rate = total_hits / num_queries
mrr = total_reciprocal_rank / num_queries if num_queries > 0 else 0

print(f"Hit Rate @ {k}: {hit_rate:.4f}")
print(f"Mean Reciprocal Rank (MRR) @ {k}: {mrr:.4f}")

### Use utility function

In [ ]:
from src.embeddings.evaluate_embeddings import calculate_performance

st_model = SentenceTransformer('all-MiniLM-L6-v2')
def embed_with_st(query_text):
    return st_model.encode([query_text])

In [ ]:
performance_os = calculate_performance(
    embeddings=all_embeddings,      # First argument should be the embeddings
    evaluation_pairs=evaluation_data,   # Second argument should be the evaluation data
    embedding_function=embed_with_st,
    k=5
)
print(f"Hit Rate @ {k}: {performance_os['hit_rate']:.4f}")
print(f"MRR @ {k}: {performance_os['mrr']:.4f}")